# Model 0: Heuristic baseline

No training. Scores candidates using domain-knowledge weights over four structured signals: skill coverage, experience gap (sigmoid-transformed), education match, and TF-IDF title similarity. Establishes what domain intuition alone achieves before any learning. Input: `outputs/features.csv`. Output: `outputs/model0_predictions.csv`.

In [1]:
# Load features.csv, re-parse list columns, group-aware split
import pandas as pd
import numpy as np
import ast
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv('../outputs/features.csv')

# Re-parse list columns saved as strings in CSV
def safe_parse(val):
    if pd.isna(val) or str(val).strip() in ('', '[]', 'nan'): return []
    try:
        r = ast.literal_eval(str(val))
        return r if isinstance(r, list) else [r]
    except Exception:
        return []

for col in ['skills', 'skills_required', 'positions']:
    df[col] = df[col].apply(safe_parse)

# Surrogate job_id for group-aware split
df['job_id'] = df['job_position_name'].factorize()[0]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['job_id']))

print(f'Train: {len(train_idx)} rows, {df.iloc[train_idx]["job_id"].nunique()} unique jobs')
print(f'Test : {len(test_idx)} rows,  {df.iloc[test_idx]["job_id"].nunique()} unique jobs')

Train: 7433 rows, 22 unique jobs
Test : 2027 rows,  6 unique jobs


## Heuristic scoring formula

Four signals combined with hand-crafted weights:

1. **`skill_coverage`** — % of required skills covered by candidate. Weight 0.4 — strongest signal.
2. **`exp_score`** — sigmoid of `exp_gap` to model threshold effect: small gaps tolerated, large gaps penalised. Weight 0.3.
3. **`edu_match`** — binary flag. Weight 0.15.
4. **`title_sim`** — TF-IDF cosine between candidate positions text and `job_position_name`. Weight 0.15.

Weights sum to 1. Set by intuition, validated by rank correlation on train set.

In [2]:
# Fit TF-IDF for title similarity, compute sigmoid exp_score, assemble heuristic predictions
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Candidate positions as joined text for TF-IDF
positions_text = df['positions'].apply(
    lambda lst: ' '.join(
        str(p) for p in lst
        if p is not None and str(p) not in ('None', 'nan', 'N/A')
    )
)
job_text = df['job_position_name'].fillna('').str.strip()

# Fit on train corpus only
tfidf = TfidfVectorizer(min_df=1, ngram_range=(1, 2))
train_corpus = positions_text.iloc[train_idx].tolist() + job_text.iloc[train_idx].tolist()
tfidf.fit(train_corpus)

pos_vecs = tfidf.transform(positions_text)
job_vecs = tfidf.transform(job_text)

title_sim = np.array([
    cosine_similarity(pos_vecs[i], job_vecs[i])[0, 0]
    for i in range(len(df))
])
df['title_sim'] = title_sim

# Sigmoid centred at 2-year gap: small gaps tolerated, large gaps penalised
df['exp_score'] = 1 / (1 + np.exp(np.abs(df['exp_gap']) - 2))

def compute_score(data, w_skill=0.40, w_exp=0.30, w_edu=0.15, w_title=0.15):
    return (
        w_skill * data['skill_coverage'] +
        w_exp   * data['exp_score'] +
        w_edu   * data['edu_match'] +
        w_title * data['title_sim']
    ).clip(0, 1)

df['model0_pred'] = compute_score(df)

print('Component stats (full dataset):')
for col in ['skill_coverage', 'exp_score', 'edu_match', 'title_sim', 'model0_pred']:
    s = df[col]
    print(f'  {col:20s}  mean={s.mean():.4f}  min={s.min():.4f}  max={s.max():.4f}')

Component stats (full dataset):
  skill_coverage        mean=0.0141  min=0.0000  max=1.0000
  exp_score             mean=0.4071  min=0.0000  max=0.8808
  edu_match             mean=0.6892  min=0.0000  max=1.0000
  title_sim             mean=0.0120  min=0.0000  max=0.7730
  model0_pred           mean=0.2330  min=0.0000  max=0.8142


## Validate weights on train set

Spearman rank correlation between heuristic score and `matched_score` on train set only.
If correlation is low, adjust weights and re-run. Document final weights chosen.

In [3]:
# Try 5 weight combinations on train, pick best Spearman
from scipy.stats import spearmanr

train_df = df.iloc[train_idx]

weight_combos = [
    dict(w_skill=0.40, w_exp=0.30, w_edu=0.15, w_title=0.15),
    dict(w_skill=0.50, w_exp=0.25, w_edu=0.15, w_title=0.10),
    dict(w_skill=0.35, w_exp=0.35, w_edu=0.15, w_title=0.15),
    dict(w_skill=0.45, w_exp=0.30, w_edu=0.10, w_title=0.15),
    dict(w_skill=0.55, w_exp=0.25, w_edu=0.10, w_title=0.10),
]

print(f'{"skill":>7} {"exp":>6} {"edu":>6} {"title":>7}  {"Spearman r":>12}')
print('─' * 48)

best_r, best_w = -1, None
for w in weight_combos:
    scores = compute_score(train_df, **w)
    r, _ = spearmanr(scores, train_df['matched_score'])
    flag = '  ← best' if r > best_r else ''
    print(f'  {w["w_skill"]:5.2f}  {w["w_exp"]:5.2f}  {w["w_edu"]:5.2f}  {w["w_title"]:5.2f}  {r:12.4f}{flag}')
    if r > best_r:
        best_r, best_w = r, w

print(f'\nFinal weights : {best_w}')
print(f'Train Spearman: {best_r:.4f}')

  skill    exp    edu   title    Spearman r
────────────────────────────────────────────────
   0.40   0.30   0.15   0.15        0.0278  ← best
   0.50   0.25   0.15   0.10        0.0273
   0.35   0.35   0.15   0.15        0.0271
   0.45   0.30   0.10   0.15        0.0465  ← best
   0.55   0.25   0.10   0.10        0.0473  ← best

Final weights : {'w_skill': 0.55, 'w_exp': 0.25, 'w_edu': 0.1, 'w_title': 0.1}
Train Spearman: 0.0473


best train Spearman after tuning 5 weight combinations: 0.0473. essentially zero. the weights that won (skill=0.55, exp=0.25, edu=0.10, title=0.10) up-weight skill coverage heavily but it fires for only 6% of pairs — the formula reduces to an experience + title signal for the other 94%. these signals alone can't track the label. this is the motivation for learned models.

In [4]:
# Evaluate test set: MAE/RMSE with rescaling, Spearman, NDCG@5 per job
from sklearn.metrics import mean_absolute_error, mean_squared_error, ndcg_score

test_df = df.iloc[test_idx].copy()
test_df['model0_pred'] = compute_score(test_df, **best_w).values

y_true = test_df['matched_score']
y_pred = test_df['model0_pred']

# Min-max rescale predictions to the range of matched_score in the train set
# Shows what the heuristic achieves if we correct for scale mismatch
train_min = df.iloc[train_idx]['matched_score'].min()
train_max = df.iloc[train_idx]['matched_score'].max()
y_pred_rescaled = (
    (y_pred - y_pred.min()) / (y_pred.max() - y_pred.min())
    * (train_max - train_min)
    + train_min
)

mae_raw       = mean_absolute_error(y_true, y_pred)
rmse_raw      = np.sqrt(mean_squared_error(y_true, y_pred))
mae_rescaled  = mean_absolute_error(y_true, y_pred_rescaled)
rmse_rescaled = np.sqrt(mean_squared_error(y_true, y_pred_rescaled))
r, _          = spearmanr(y_pred, y_true)

print('Pointwise metrics (test set)')
print(f'  {"":22s}  {"original":>10}  {"rescaled":>10}')
print(f'  {"MAE":22s}  {mae_raw:10.4f}  {mae_rescaled:10.4f}')
print(f'  {"RMSE":22s}  {rmse_raw:10.4f}  {rmse_rescaled:10.4f}')
print(f'  {"Spearman r":22s}  {r:10.4f}  {"(unchanged)":>10}')
print(f'  train score range: [{train_min:.3f}, {train_max:.3f}]')
print(f'  raw pred range   : [{y_pred.min():.3f}, {y_pred.max():.3f}]')
print()

# NDCG@5 grouped by job
ndcg_results = []
for _, group in test_df.groupby('job_id'):
    if len(group) < 2:
        continue
    true = group['matched_score'].values.reshape(1, -1)
    pred = group['model0_pred'].values.reshape(1, -1)
    ndcg_results.append({
        'job'  : group['job_position_name'].iloc[0],
        'ndcg5': ndcg_score(true, pred, k=5),
        'n'    : len(group),
    })

mean_ndcg = np.mean([x['ndcg5'] for x in ndcg_results])
print(f'NDCG@5 mean  : {mean_ndcg:.4f}  ({len(ndcg_results)} jobs)')
print()
print('Per-job NDCG@5 (sorted descending):')
for row in sorted(ndcg_results, key=lambda x: -x["ndcg5"]):
    print(f'  {row["ndcg5"]:.4f}  n={row["n"]:3d}  {row["job"][:60]}')

Pointwise metrics (test set)
                            original    rescaled
  MAE                         0.5560      0.4392
  RMSE                        0.5815      0.4805
  Spearman r                  0.1653  (unchanged)
  train score range: [0.040, 0.970]
  raw pred range   : [0.000, 0.595]

NDCG@5 mean  : 0.8555  (6 jobs)

Per-job NDCG@5 (sorted descending):
  0.8989  n=338  Head of Internal Control & Compliance (ICC) - SEVP/DMD
  0.8773  n=338  Manager- Human Resource Management (HRM)
  0.8447  n=337  Executive/ Sr. Executive -IT
  0.8404  n=338  Asst. Manager/ Manger (Administrative)
  0.8368  n=338  Senior Software Engineer
  0.8349  n=338  Database Administrator (DBA)


MAE 0.439 (rescaled), RMSE 0.481, Spearman 0.165. the rescaling is necessary because raw heuristic scores peak at 0.595 while the true score distribution reaches 0.97 — the signal exists but the scale is completely wrong. NDCG@5 of 0.856 looks acceptable but is misleading: 338 candidates per job with most scores between 0.5–0.85 means even weak ranking scores high. the real failure is the near-zero Spearman — the heuristic cannot rank candidates in the right order across jobs.

In [5]:
# Save test predictions to outputs/model0_predictions.csv
import os
os.makedirs('../outputs', exist_ok=True)

out_cols = ['job_position_name', 'candidate_doc', 'job_doc', 'matched_score', 'model0_pred']
test_df[out_cols].to_csv('../outputs/model0_predictions.csv', index=False)

print(f'Saved {len(test_df)} rows → ../outputs/model0_predictions.csv')
print()
print(test_df[out_cols].head(3)[['job_position_name','matched_score','model0_pred']].to_string())

Saved 2027 rows → ../outputs/model0_predictions.csv

                         job_position_name  matched_score  model0_pred
0                 Senior Software Engineer           0.85     0.167235
11  Asst. Manager/ Manger (Administrative)           0.65     0.225000
15            Database Administrator (DBA)           0.85     0.303571
